# 02 — Demand preparation and data audit

Parse the immutable field sheets, inspect the normalized records, and verify the node × route × time demand tensor. Complete this notebook before interpreting any equilibrium result.

## Data rule

`data/raw/` is the source record. The normalized CSV in `data/processed/` is generated, so correct a survey issue in the source record and rerun this notebook rather than editing the generated CSV.

In [ ]:
from pathlib import Path
import os
import pandas as pd

root = Path.cwd()
if root.name == 'notebooks':
    root = root.parent
os.chdir(root)

from src.config import MODEL_ROUTES, NODES, ROUTES, WINDOWS
from src.data_parser import build_boarding_counts
from src.demand import build_demand_tensor, load_field_sheets
from src.paths import ProjectPaths

paths = ProjectPaths.discover()

In [ ]:
raw_files = sorted(paths.raw_data.glob('DATA-*.csv'))
assert raw_files, f'No DATA-*.csv files found in {paths.raw_data}'
pd.DataFrame({'field_survey_file': [file.name for file in raw_files]})

In [ ]:
build_boarding_counts(paths.raw_data, paths.processed_data)
field_data = load_field_sheets(paths.processed_data)
print(f'{len(field_data):,} normalized observations')
field_data.head()

## Quality checks

The checks below catch unknown labels, non-positive durations, and missing observed combinations. A missing combination is reported for review; it is not silently imputed.

In [ ]:
assert set(field_data['node']).issubset(NODES)
assert set(field_data['route']).issubset(ROUTES)
assert set(field_data['window']).issubset(WINDOWS)
assert (field_data['passengers_boarding'] >= 0).all()
assert (field_data['obs_duration_min'] > 0).all()

summary = (field_data.groupby(['node', 'route', 'window'], observed=True)
           .agg(observations=('date', 'size'), boardings=('passengers_boarding', 'sum'))
           .reset_index())
summary.sort_values(['node', 'route', 'window']).head(15)

In [ ]:
expected = pd.MultiIndex.from_product([NODES, ROUTES, WINDOWS], names=['node', 'route', 'window'])
observed = pd.MultiIndex.from_frame(summary[['node', 'route', 'window']])
missing = expected.difference(observed)
pd.DataFrame(list(missing), columns=['node', 'route', 'window'])

## Demand tensor

Axis order is fixed: `[node, route, window]`. The `EXTERNAL` column is retained for auditability but excluded from the seven-route allocation model.

In [ ]:
demand = build_demand_tensor(field_data)
assert demand.shape == (len(NODES), len(ROUTES), len(WINDOWS))

route_demand = pd.DataFrame(demand.sum(axis=0), index=ROUTES, columns=WINDOWS)
route_demand['total_passengers_per_window'] = route_demand.sum(axis=1)
# EXTERNAL is retained for data audit but excluded from the seven-route model.
route_demand[route_demand.index != 'EXTERNAL']

In [ ]:
node_demand = pd.DataFrame(demand[:, :len(MODEL_ROUTES), :].sum(axis=1), index=NODES, columns=WINDOWS)
node_demand['total_passengers_per_window'] = node_demand.sum(axis=1)
node_demand